# MBRAIN Pipeline: Processing a New Mouse Subject

This notebook walks through **every step** of the MBRAIN tractography pipeline for processing
a new mouse subject, from raw BIDS data to final cleaned tracts and metrics.

**How to use this notebook:**
1. Fill in the subject info in Section 0
2. Run each cell sequentially
3. At each QC checkpoint (🔍), inspect the output and paste a screenshot
4. Some steps have **CTRL** vs **AP/APTR** branches — run only the one that applies

**Pipeline Overview:**
```
0. Setup & Config
1. T2w Anatomical Processing (flip1 transformation)
2. DWI Preprocessing (flip, denoise, bias correct, register to T2w)
3. ROI Preparation
   ├── CTRL: Allen Atlas Coregistration + ROI Extraction
   └── AP/APTR: Manual ROI flip + hemisphere correction
4. FOD Estimation (response functions, multi-tissue CSD, normalization)
5. ROI Resampling to FOD space
6. Segmentation
   ├── CTRL: Isocortex → 5 A-P segments
   └── AP/APTR: Graft → 9 segments (3 A-P × 3 D-V)
7. Whole-Brain Tractography (1M streamlines)
8. Passthrough Tract Extraction (tckedit with -include)
9. MATLAB AFQ Cleaning
10. SIFT2 Connectivity Weights
11. Metrics Report Generation
```

---

## Step 0: Setup & Configuration

Load the config and set up all paths. **Edit the cell below** with your subject info.

In [32]:
# ============================================================
# EDIT THIS CELL: Fill in your subject information
# ============================================================

SUBJECT_ID = "AP_test_mouseNmM"        # e.g. "sub-129-921-R1F", "sub-Apallial17NmF"
SESSION_ID = "Pasca_769-C-OO-NmM_9-11-24 - 240911111853"          # e.g. "ses-260115121231"
GROUP      = "AP"                  # "CTRL", "AP", or "APTR"

# For AP/APTR only: segmentation mode
SEG_MODE = 9  # 6 (3 A-P × 2 D-V) or 9 (3 A-P × 3 D-V) — recommended: 9

In [33]:
import os, sys, subprocess, yaml
from pathlib import Path
from IPython.display import display, HTML, Image, Markdown

# Add project root to path
REPO_DIR = Path("/Users/glerma/soft/mbrain")
sys.path.insert(0, str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR / "functions"))

# Load config
from functions.config_loader import load_config
cfg = load_config()

# Clean subject and session IDs for BIDS paths
from utils.file_utils import clean_subject_id, extract_session_info
cleaned_subject = clean_subject_id(SUBJECT_ID.replace("sub-", ""))
_, cleaned_session = extract_session_info(SESSION_ID)

# Derived paths
DERIV = Path(cfg['paths']['derivatives_dir'])
BIDS  = Path(cfg['paths']['bids_dir'])
ATLAS = Path(cfg['paths']['atlas_dir'])
BASE  = DERIV / SUBJECT_ID / SESSION_ID
DWI   = BASE / "dwi"
BIDS_SUBJ_DIR = BIDS / f"sub-{cleaned_subject}" / f"ses-{cleaned_session}"
BIDS_DWI  = BIDS_SUBJ_DIR / "dwi"
BIDS_ANAT = BIDS_SUBJ_DIR / "anat"

# Tractography params from config
tck = cfg['tractography']
mat = cfg['matlab_cleaning']

print(f"Subject:    {SUBJECT_ID} → sub-{cleaned_subject}")
print(f"Session:    {SESSION_ID} → ses-{cleaned_session}")
print(f"Group:      {GROUP}")
print(f"Base path:  {BASE}")
print(f"BIDS DWI:   {BIDS_DWI}")
print(f"BIDS Anat:  {BIDS_ANAT}")
print(f"Tractography: algo={tck['algorithm']}, select={tck['select']}, step={tck['step_size']}, angle={tck['angle']}, cutoff={tck['cutoff']}")

Subject:    AP_test_mouseNmM → sub-APtestmouseNmM
Session:    Pasca_769-C-OO-NmM_9-11-24 - 240911111853 → ses-240911111853
Group:      AP
Base path:  /Users/glerma/soft/mbrain/data/derivatives/mrtrix_preproc_flip1/AP_test_mouseNmM/Pasca_769-C-OO-NmM_9-11-24 - 240911111853
BIDS DWI:   /Users/glerma/soft/mbrain/data/bids/sub-APtestmouseNmM/ses-240911111853/dwi
BIDS Anat:  /Users/glerma/soft/mbrain/data/bids/sub-APtestmouseNmM/ses-240911111853/anat
Tractography: algo=iFOD2, select=2000, step=0.02, angle=60, cutoff=0.1


In [34]:
# Helper: run a shell command and print output
def run(cmd, check=True, timeout=None):
    """Run a shell command, print it, and return the result."""
    print(f"\n>>> {cmd}\n")
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print("STDERR:", result.stderr.strip())
    if check and result.returncode != 0:
        print(f"⚠️ Command failed with exit code {result.returncode}")
    return result

# Helper: check if file exists and show info
def check_file(path, label=""):
    p = Path(path)
    if p.exists():
        size = p.stat().st_size / (1024*1024)
        print(f"✅ {label or p.name}: {size:.1f} MB")
        return True
    else:
        print(f"❌ MISSING: {label or p.name} ({p})")
        return False

# Helper: run mrinfo
def mrinfo(path):
    run(f"mrinfo {path}")

print("Helpers loaded ✅")

Helpers loaded ✅


## Step 0b: Flywheel → BIDS Conversion

If your subject's raw data is still in Flywheel format (`local/flywheel/fMRI/SUBJECTS/`),
run this step to convert it to BIDS format before proceeding.

**Flywheel layout:**
```
local/flywheel/fMRI/SUBJECTS/{SubjectName}/SESSIONS/{SessionName}/ACQUISITIONS/
    ├── DTI_EPI_seg_30dir_sat/FILES/   → .nii.gz, .bval, .mvec, .json
    ├── 4_T2w_lowres/FILES/            → .nii.gz, .json
    ├── 5_T2w_highres/FILES/           → .nii.gz, .json
    ├── T2star_FID_EPI.../FILES/       → .nii.gz, .json
    └── 1_Localizer/FILES/             → .nii.gz, .json
```

**BIDS output:**
```
data/bids/sub-{ID}/ses-{SessionID}/
    ├── anat/   → _acq-lowres_T2w.nii.gz, _acq-highres_T2w.nii.gz, _T2starw.nii.gz, ...
    └── dwi/    → _dwi.nii.gz, _dwi.bval, _dwi.bvec, _dwi.json
```

**Skip this cell if your subject is already in `data/bids/`.**

In [35]:
# ============================================================
# Flywheel → BIDS Conversion (uses utils/bids_converter.py)
# Set DRY_RUN=True to preview, False to actually copy files
# Skip this cell if subject is already in data/bids/
# ============================================================
from utils.bids_converter import FlywheelToBIDSConverter

DRY_RUN = False  # ← Change to False to actually copy files

FW_ROOT = REPO_DIR / "local" / "flywheel" / "fMRI"
raw_subject_name = SUBJECT_ID.replace("sub-", "")
FW_SUBJECT = FW_ROOT / "SUBJECTS" / raw_subject_name

if not FW_SUBJECT.exists():
    print(f"⚠️  Flywheel source not found: {FW_SUBJECT}")
    print(f"   Available: {[d.name for d in (FW_ROOT / 'SUBJECTS').iterdir() if d.is_dir() and not d.name.startswith('.')]}")
    print(f"   If already in BIDS, skip this cell.")
else:
    converter = FlywheelToBIDSConverter(flywheel_root=FW_ROOT, bids_root=BIDS)
    converter._convert_subject(FW_SUBJECT, dry_run=DRY_RUN)
    
    if DRY_RUN:
        print("\n🔍 DRY RUN — no files copied. Set DRY_RUN = False to convert.")
    else:
        print(f"\n✅ Done! Files copied: {converter.stats['files_copied']}")
        if converter.stats['errors']:
            print(f"⚠️ Errors: {converter.stats['errors']}")

2026-02-14 19:57:42,580 - INFO - Processing subject: APtestmouseNmM
2026-02-14 19:57:42,634 - INFO -   Processing session: 240911111853 (date: 20240911)
2026-02-14 19:57:42,635 - INFO -     Processing acquisition: DTI_EPI_seg_30dir_sat
2026-02-14 19:57:42,645 - INFO -       Copied: sub-APtestmouseNmM_ses-240911111853_dwi.nii.gz
2026-02-14 19:57:42,649 - INFO -       Copied: sub-APtestmouseNmM_ses-240911111853_dwi.json
2026-02-14 19:57:42,651 - INFO -       Copied: sub-APtestmouseNmM_ses-240911111853_dwi.bval
2026-02-14 19:57:42,655 - INFO -       Copied: sub-APtestmouseNmM_ses-240911111853_dwi.bvec
2026-02-14 19:57:42,656 - INFO -     Processing acquisition: 4_T2w_lowres
2026-02-14 19:57:42,657 - INFO -       Copied: sub-APtestmouseNmM_ses-240911111853_acq-lowres_T2w.nii.gz
2026-02-14 19:57:42,658 - INFO -       Copied: sub-APtestmouseNmM_ses-240911111853_acq-lowres_T2w.json
2026-02-14 19:57:42,659 - INFO -     Processing acquisition: T2star_FID_EPI_300KHz_200micron_1
2026-02-14 19:57:


✅ Done! Files copied: 16


In [36]:
# 🔍 QC: Verify BIDS conversion output
print("--- BIDS output verification ---")
run(f"find {BIDS_SUBJ_DIR} -type f -not -name '.*' | sort", check=False)

print("\n--- Critical files ---")
bids_prefix = f"sub-{cleaned_subject}_ses-{cleaned_session}"
check_file(BIDS_DWI / f"{bids_prefix}_dwi.nii.gz", "DWI NIfTI")
check_file(BIDS_DWI / f"{bids_prefix}_dwi.bval", "DWI bval")
check_file(BIDS_DWI / f"{bids_prefix}_dwi.bvec", "DWI bvec")
check_file(BIDS_ANAT / f"{bids_prefix}_acq-highres_T2w.nii.gz", "T2w highres")
check_file(BIDS_ANAT / f"{bids_prefix}_acq-lowres_T2w.nii.gz", "T2w lowres")

--- BIDS output verification ---

>>> find /Users/glerma/soft/mbrain/data/bids/sub-APtestmouseNmM/ses-240911111853 -type f -not -name '.*' | sort

/Users/glerma/soft/mbrain/data/bids/sub-APtestmouseNmM/ses-240911111853/anat/sub-APtestmouseNmM_ses-240911111853_T2starw.json
/Users/glerma/soft/mbrain/data/bids/sub-APtestmouseNmM/ses-240911111853/anat/sub-APtestmouseNmM_ses-240911111853_T2starw.nii.gz
/Users/glerma/soft/mbrain/data/bids/sub-APtestmouseNmM/ses-240911111853/anat/sub-APtestmouseNmM_ses-240911111853_acq-highres_T2w.json
/Users/glerma/soft/mbrain/data/bids/sub-APtestmouseNmM/ses-240911111853/anat/sub-APtestmouseNmM_ses-240911111853_acq-highres_T2w.nii.gz
/Users/glerma/soft/mbrain/data/bids/sub-APtestmouseNmM/ses-240911111853/anat/sub-APtestmouseNmM_ses-240911111853_acq-localizer_T2w.json
/Users/glerma/soft/mbrain/data/bids/sub-APtestmouseNmM/ses-240911111853/anat/sub-APtestmouseNmM_ses-240911111853_acq-localizer_T2w.nii.gz
/Users/glerma/soft/mbrain/data/bids/sub-APtestmouseNmM/

True

In [37]:
# Create output directories
os.makedirs(DWI, exist_ok=True)
print(f"Output directory ready: {DWI}")

# Verify BIDS source files exist
print("\n--- Checking BIDS source files ---")
print("\nAnatomy:")
run(f"ls -la {BIDS_ANAT}/", check=False)
print("\nDWI:")
run(f"ls -la {BIDS_DWI}/", check=False)

Output directory ready: /Users/glerma/soft/mbrain/data/derivatives/mrtrix_preproc_flip1/AP_test_mouseNmM/Pasca_769-C-OO-NmM_9-11-24 - 240911111853/dwi

--- Checking BIDS source files ---

Anatomy:

>>> ls -la /Users/glerma/soft/mbrain/data/bids/sub-APtestmouseNmM/ses-240911111853/anat/

total 270608
drwxr-xr-x@ 10 glerma  staff        320 Feb 14 19:31 .
drwxr-xr-x@  4 glerma  staff        128 Feb 14 19:31 ..
-rw-------@  1 glerma  staff       1322 Dec 17 21:58 sub-APtestmouseNmM_ses-240911111853_T2starw.json
-rw-------@  1 glerma  staff  135443817 Dec 17 21:58 sub-APtestmouseNmM_ses-240911111853_T2starw.nii.gz
-rw-------@  1 glerma  staff       1308 Dec 17 21:58 sub-APtestmouseNmM_ses-240911111853_acq-highres_T2w.json
-rw-------@  1 glerma  staff    1848722 Dec 17 21:58 sub-APtestmouseNmM_ses-240911111853_acq-highres_T2w.nii.gz
-rw-------@  1 glerma  staff       1170 Dec 17 21:58 sub-APtestmouseNmM_ses-240911111853_acq-localizer_T2w.json
-rw-------@  1 glerma  staff     948145 Dec 17 2

CompletedProcess(args='ls -la /Users/glerma/soft/mbrain/data/bids/sub-APtestmouseNmM/ses-240911111853/dwi/', returncode=0, stdout='total 36552\ndrwxr-xr-x@ 6 glerma  staff       192 Feb 14 19:31 \x1b.\x1b[m\x1b[m\ndrwxr-xr-x@ 4 glerma  staff       128 Feb 14 19:31 \x1b..\x1b[m\x1b[m\n-rw-------@ 1 glerma  staff       357 Dec 17 21:58 sub-APtestmouseNmM_ses-240911111853_dwi.bval\n-rw-------@ 1 glerma  staff      1265 Dec 17 21:58 sub-APtestmouseNmM_ses-240911111853_dwi.bvec\n-rw-------@ 1 glerma  staff      1303 Dec 17 21:58 sub-APtestmouseNmM_ses-240911111853_dwi.json\n-rw-------@ 1 glerma  staff  18699483 Dec 17 21:58 sub-APtestmouseNmM_ses-240911111853_dwi.nii.gz\n', stderr='')

---
## Step 1: BIDS to MRtrix Preprocessing (T2w + DWI)

Apply the flip1 transformation and full DWI preprocessing pipeline:
1. **T2w processing**: Axis permutation `(2,1,0)` + flip axis 1 (Y)
2. **DWI processing**: Convert → Flip1 → Gradient check → Denoise → Mask → Bias correct → Register to T2w

In [38]:
# ============================================================
# BIDS to MRtrix Preprocessing (T2w + DWI)
# ============================================================

print("Running BIDS → MRtrix preprocessing...")
print("This includes T2w flipping and full DWI preprocessing pipeline.")
print("Steps: T2w flip1 transformation + DWI convert/flip/denoise/bias-correct/register")
print()

# Run the preprocessing using batch_process_flip1.py
# This script handles all the flip1 pipeline steps automatically
#
# OPTIONS for batch_process_flip1.py:
#   --subject SUBJECT_ID    : Process single subject (required if not using --group)
#   --group GROUP           : Process all subjects in group (AP, APTR, or CTRL)
#   --steps STEPS           : Comma-separated steps to run (default: 1,2,3,4,5,6,7,8)
#                           : 1=create folders, 2=flip T2w, 3=flip ROIs, 4=preprocess DWI,
#                           : 5=generate FOD, 6=resample ROIs, 7=create segments, 8=tractography
#   --dry-run               : Preview what would be done without executing
#   --force                 : Force reprocessing even if files exist
#
# EXAMPLES:
#   python functions/batch_process_flip1.py --subject sub-Apallial20R1F
#   python functions/batch_process_flip1.py --group AP --dry-run
#   python functions/batch_process_flip1.py --group CTRL --steps 2,4
#   python functions/batch_process_flip1.py --subject sub-129-921-R1F --force

result = run(
    f"cd {REPO_DIR} && micromamba run -n mbrain python functions/batch_process_flip1.py "
    f"--subject {SUBJECT_ID} --steps 2,4 --dry-run ",
    timeout=1800
)
success = result.returncode == 0

# Check results
if success:
    print("✅ Preprocessing completed successfully!")
else:
    print("❌ Preprocessing failed!")
    print(f"Return code: {result.returncode}")
    if result.stderr:
        print(f"STDERR: {result.stderr}")

# Verify outputs (same for both options)
print("\n--- Verifying preprocessing outputs ---")
check_file(DWI / "t2w_highres_flip.mif", "T2w high-res (flipped)")
check_file(DWI / "t2w_lowres_flip.mif", "T2w low-res (flipped)")  
check_file(DWI / "dwi_flip_gradcheck_denoise_bias_anat.mif", "Final preprocessed DWI")
check_file(DWI / "mask_anat.mif", "Brain mask (anat space)")
check_file(DWI / "b0_final.mif", "Final b0 reference")

Running BIDS → MRtrix preprocessing...
This includes T2w flipping and full DWI preprocessing pipeline.
Steps: T2w flip1 transformation + DWI convert/flip/denoise/bias-correct/register


>>> cd /Users/glerma/soft/mbrain && micromamba run -n mbrain python functions/batch_process_flip1.py --subject AP_test_mouseNmM --steps 2,4 --dry-run 

[2026-02-14 19:58:41] [ERROR] No subjects found to process
⚠️ Command failed with exit code 1
❌ Preprocessing failed!
Return code: 1

--- Verifying preprocessing outputs ---
❌ MISSING: T2w high-res (flipped) (/Users/glerma/soft/mbrain/data/derivatives/mrtrix_preproc_flip1/AP_test_mouseNmM/Pasca_769-C-OO-NmM_9-11-24 - 240911111853/dwi/t2w_highres_flip.mif)
❌ MISSING: T2w low-res (flipped) (/Users/glerma/soft/mbrain/data/derivatives/mrtrix_preproc_flip1/AP_test_mouseNmM/Pasca_769-C-OO-NmM_9-11-24 - 240911111853/dwi/t2w_lowres_flip.mif)
❌ MISSING: Final preprocessed DWI (/Users/glerma/soft/mbrain/data/derivatives/mrtrix_preproc_flip1/AP_test_mouseNmM/Pasca_

False

In [39]:
# Open T2w in mrview for visual QC
run(f"mrview {DWI}/t2w_highres_flip.mif &", check=False)
print("\n👆 Check orientation in mrview. Close when done.")


>>> mrview /Users/glerma/soft/mbrain/data/derivatives/mrtrix_preproc_flip1/AP_test_mouseNmM/Pasca_769-C-OO-NmM_9-11-24 - 240911111853/dwi/t2w_highres_flip.mif &

STDERR: mrview: [ERROR] unknown format for image "/Users/glerma/soft/mbrain/data/derivatives/mrtrix_preproc_flip1/AP_test_mouseNmM/Pasca_769-C-OO-NmM_9-11-24"
mrview: [ERROR] error opening image "/Users/glerma/soft/mbrain/data/derivatives/mrtrix_preproc_flip1/AP_test_mouseNmM/Pasca_769-C-OO-NmM_9-11-24"
mrview: [ERROR] no filename supplied to standard input (broken pipe?)
mrview: [ERROR] error opening image "-"
mrview: [ERROR] failed to open key/value file "240911111853/dwi/t2w_highres_flip.mif": No such file or directory
mrview: [ERROR] error opening image "240911111853/dwi/t2w_highres_flip.mif"

👆 Check orientation in mrview. Close when done.


**QC 1 Screenshot:**

_Paste screenshot here_

**QC 1 Result:** ☐ PASS / ☐ FAIL

---

### 🔍 QC Checkpoint 1: T2w Orientation

Run the mrview command below, then paste a screenshot here.

**Verify:**
- Dorsal (top of brain) is UP
- Ventral (bottom) is DOWN
- Anterior (nose) is to the RIGHT in sagittal view
- Left hemisphere appears on anatomical LEFT

In [40]:
# Open T2w in mrview for visual QC
run(f"mrview {DWI}/t2w_highres_flip.mif &", check=False)
print("\n👆 Check orientation in mrview. Close when done.")


>>> mrview /Users/glerma/soft/mbrain/data/derivatives/mrtrix_preproc_flip1/AP_test_mouseNmM/Pasca_769-C-OO-NmM_9-11-24 - 240911111853/dwi/t2w_highres_flip.mif &

STDERR: mrview: [ERROR] unknown format for image "/Users/glerma/soft/mbrain/data/derivatives/mrtrix_preproc_flip1/AP_test_mouseNmM/Pasca_769-C-OO-NmM_9-11-24"
mrview: [ERROR] error opening image "/Users/glerma/soft/mbrain/data/derivatives/mrtrix_preproc_flip1/AP_test_mouseNmM/Pasca_769-C-OO-NmM_9-11-24"
mrview: [ERROR] no filename supplied to standard input (broken pipe?)
mrview: [ERROR] error opening image "-"
mrview: [ERROR] failed to open key/value file "240911111853/dwi/t2w_highres_flip.mif": No such file or directory
mrview: [ERROR] error opening image "240911111853/dwi/t2w_highres_flip.mif"

👆 Check orientation in mrview. Close when done.


**QC 1 Screenshot:**

_Paste screenshot here_

**QC 1 Result:** ☐ PASS / ☐ FAIL

---

### 🔍 QC Checkpoint 2: DWI-T2w Registration

**Verify:** The b0 image should be perfectly aligned with the T2w.

In [41]:
# View b0 overlaid on T2w to check registration
run(f"mrview {DWI}/t2w_highres_flip.mif "
    f"-overlay.load {DWI}/b0_final.mif -overlay.opacity 0.5 &", check=False)
print("\n👆 Verify b0 aligns with T2w. Brain edges should match.")


>>> mrview /Users/glerma/soft/mbrain/data/derivatives/mrtrix_preproc_flip1/AP_test_mouseNmM/Pasca_769-C-OO-NmM_9-11-24 - 240911111853/dwi/t2w_highres_flip.mif -overlay.load /Users/glerma/soft/mbrain/data/derivatives/mrtrix_preproc_flip1/AP_test_mouseNmM/Pasca_769-C-OO-NmM_9-11-24 - 240911111853/dwi/b0_final.mif -overlay.opacity 0.5 &

STDERR: mrview: [ERROR] options must appear after the last argument - see help page for details

👆 Verify b0 aligns with T2w. Brain edges should match.


**QC 2 Screenshot:**

_Paste screenshot here_

**QC 2 Result:** ☐ PASS / ☐ FAIL

---

## Step 2: ROI Preparation

This step differs by group:
- **CTRL**: Register Allen CCF atlas to subject T2w, extract ROIs
- **AP/APTR**: Flip manually-drawn ROIs, correct hemisphere labels

**Run only the section that matches your group.**

### Step 2A: CTRL — Allen Atlas Coregistration + ROI Extraction

⚠️ **Skip this if your subject is AP or APTR**

In [ ]:
# ============================================================
# CTRL ONLY: Atlas coregistration and ROI extraction
# ============================================================
if GROUP == "CTRL":
    print("Running atlas ROI extraction for CTRL subject...")
    print("This takes ~5 minutes for ANTs registration.\n")
    
    result = run(
        f"cd {REPO_DIR} && micromamba run -n mbrain python functions/atlas_roi_extraction.py "
        f"--subject {SUBJECT_ID} --session {SESSION_ID}",
        timeout=600
    )
    
    # Check outputs
    atlas_dir = BASE / "atlas"
    rois_dir  = atlas_dir / "rois"
    print("\n--- Atlas ROI outputs ---")
    check_file(atlas_dir / "annotation_warped.nii.gz", "Warped annotation")
    for roi in ["isocortex", "striatum", "thalamus", "cerebellum", "olfactory_cortex"]:
        check_file(rois_dir / f"roi_{roi}_left.nii.gz", f"{roi} L")
        check_file(rois_dir / f"roi_{roi}_right.nii.gz", f"{roi} R")
else:
    print(f"Skipping Step 3A (GROUP={GROUP}, not CTRL)")

### Step 2B: AP/APTR — Manual ROI Flip + Hemisphere Correction

⚠️ **Skip this if your subject is CTRL**

Manual ROIs from 3D Slicer need:
1. Flip1 transformation (same as T2w)
2. Left/Right label swap (original Slicer labels were inverted)

**Prerequisite:** Manual ROIs must exist in `{subject}/{session}/manual_roi/`

In [ ]:
# ============================================================
# AP/APTR ONLY: Manual ROI flip and hemisphere correction
# ============================================================
if GROUP in ["AP", "APTR"]:
    MANUAL_ROI_SRC = BASE / "manual_roi"
    MANUAL_ROI_FLIP = BASE / "manual_roi_flip"
    os.makedirs(MANUAL_ROI_FLIP, exist_ok=True)
    
    # ROI names for AP/APTR
    roi_names = ["Graft", "Striatum", "Thalamus", "Palecortex", "cerebellum"]
    
    print("--- Checking source manual ROIs ---")
    for name in roi_names:
        for hemi in ["left", "right"]:
            src = MANUAL_ROI_SRC / f"roi_{name}_{hemi}.nii.gz"
            check_file(src, f"{name} {hemi}")
    
    print("\n--- Flipping ROIs (with L/R swap) ---")
    for name in roi_names:
        for src_hemi, dst_hemi in [("left", "right"), ("right", "left")]:
            # Swap L/R labels during flip (Slicer L/R were inverted)
            src = MANUAL_ROI_SRC / f"roi_{name}_{src_hemi}.nii.gz"
            if not src.exists():
                print(f"  ⚠️ Skipping {src.name} (not found)")
                continue
            
            # Convert + permute + flip
            tmp_perm = MANUAL_ROI_FLIP / f"tmp_perm_{name}_{dst_hemi}.mif"
            dst = MANUAL_ROI_FLIP / f"roi_{name}_{dst_hemi}_flip.mif"
            
            run(f"mrconvert {src} {tmp_perm} -axes 2,1,0 -strides 1,2,3 -force", check=False)
            run(f"mrtransform {tmp_perm} {dst} -flip 1 -force", check=False)
            tmp_perm.unlink(missing_ok=True)
            check_file(dst, f"{name} {dst_hemi} (flipped)")
    
    print("\n✅ Manual ROIs flipped with hemisphere correction")
else:
    print(f"Skipping Step 3B (GROUP={GROUP}, not AP/APTR)")

### 🔍 QC Checkpoint 3: ROI Alignment on T2w

**Verify:** ROIs should match anatomical structures on T2w.

In [ ]:
# QC: View ROIs on T2w
if GROUP == "CTRL":
    rois_dir = BASE / "atlas" / "rois"
    cmd = (f"mrview {DWI}/t2w_highres_flip.mif "
           f"-overlay.load {rois_dir}/roi_isocortex_left.nii.gz -overlay.opacity 0.4 "
           f"-overlay.load {rois_dir}/roi_thalamus_left.nii.gz -overlay.opacity 0.4 "
           f"-overlay.load {rois_dir}/roi_striatum_left.nii.gz -overlay.opacity 0.4 &")
else:  # AP/APTR
    mroi = BASE / "manual_roi_flip"
    cmd = (f"mrview {DWI}/t2w_highres_flip.mif "
           f"-overlay.load {mroi}/roi_Graft_left_flip.mif -overlay.opacity 0.4 "
           f"-overlay.load {mroi}/roi_Thalamus_left_flip.mif -overlay.opacity 0.4 "
           f"-overlay.load {mroi}/roi_Striatum_left_flip.mif -overlay.opacity 0.4 &")

run(cmd, check=False)
print("\n👆 Check that ROIs sit on correct anatomical structures.")
print("   Left-labeled ROIs should appear on anatomical LEFT.")

**QC 3 Screenshot:**

_Paste screenshot here_

**QC 3 Result:** ☐ PASS / ☐ FAIL

---

## Step 3: FOD Estimation

Generate Fiber Orientation Distributions for tractography:
1. Response function estimation (dhollander multi-tissue)
2. FOD estimation (msmt_csd)
3. Normalization (mtnormalise — WM+CSF only, GM often fails for mice)

In [ ]:
# Use the existing script
print("Running FOD estimation (this may take several minutes)...\n")

result = run(
    f"cd {REPO_DIR} && micromamba run -n mbrain python functions/dwi_fod.py "
    f"--subject {SUBJECT_ID} --force",
    timeout=1800
)

# Check outputs
print("\n--- FOD outputs ---")
for f in ["response_wm.txt", "response_gm.txt", "response_csf.txt",
          "wm_fod.mif", "gm_fod.mif", "csf_fod.mif",
          "wm_fod_norm.mif", "csf_fod_norm.mif"]:
    check_file(DWI / f)

### 🔍 QC Checkpoint 4: FOD Quality

**Verify:** FOD glyphs should show directionally consistent orientations in WM.

In [ ]:
# View FOD glyphs on T2w
run(f"mrview {DWI}/t2w_highres_flip.mif -odf.load_sh {DWI}/wm_fod_norm.mif &", check=False)
print("\n👆 Check FOD glyphs. They should be:
"
      "   - Elongated in WM tracts (not spherical)\n"
      "   - Consistent direction along known tracts (corpus callosum, etc.)\n"
      "   - Small/absent in ventricles and outside brain")

**QC 4 Screenshot:**

_Paste screenshot here_

**QC 4 Result:** ☐ PASS / ☐ FAIL

---

## Step 4: Resample ROIs to FOD Space

ROIs are in T2w resolution but tractography needs them in FOD resolution.
Uses `mrgrid regrid` with nearest-neighbor interpolation.

In [ ]:
# Resample all ROIs to FOD space
print("Resampling ROIs to FOD space...\n")

result = run(
    f"cd {REPO_DIR} && micromamba run -n mbrain python functions/resample_rois_to_fod.py "
    f"--subject {SUBJECT_ID} --force",
    timeout=300
)

# Check outputs
print("\n--- Resampled ROI outputs ---")
if GROUP == "CTRL":
    fod_rois = BASE / "atlas" / "rois" / "fod_space"
    for roi in ["isocortex", "striatum", "thalamus", "cerebellum", "olfactory_cortex"]:
        check_file(fod_rois / f"roi_{roi}_left.nii.gz", f"{roi} L (FOD)")
        check_file(fod_rois / f"roi_{roi}_right.nii.gz", f"{roi} R (FOD)")
else:  # AP/APTR
    fod_rois = BASE / "manual_roi" / "fod_space"
    for roi in ["Graft", "Striatum", "Thalamus", "Palecortex", "cerebellum"]:
        check_file(fod_rois / f"roi_{roi}_left_flip_fod.mif", f"{roi} L (FOD)")
        check_file(fod_rois / f"roi_{roi}_right_flip_fod.mif", f"{roi} R (FOD)")

### 🔍 QC Checkpoint 5: ROIs in FOD Space

**Verify:** Resampled ROIs should still align with anatomy on the FOD grid.

In [ ]:
# View FOD-space ROIs overlaid on FOD
if GROUP == "CTRL":
    fod_rois = BASE / "atlas" / "rois" / "fod_space"
    cmd = (f"mrview {DWI}/wm_fod_norm.mif "
           f"-overlay.load {fod_rois}/roi_isocortex_left.nii.gz -overlay.opacity 0.5 "
           f"-overlay.load {fod_rois}/roi_thalamus_left.nii.gz -overlay.opacity 0.5 &")
else:
    fod_rois = BASE / "manual_roi" / "fod_space"
    cmd = (f"mrview {DWI}/wm_fod_norm.mif "
           f"-overlay.load {fod_rois}/roi_Graft_left_flip_fod.mif -overlay.opacity 0.5 "
           f"-overlay.load {fod_rois}/roi_Thalamus_left_flip_fod.mif -overlay.opacity 0.5 &")

run(cmd, check=False)
print("\n👆 Check that ROIs align on the FOD grid.")

**QC 5 Screenshot:**

_Paste screenshot here_

**QC 5 Result:** ☐ PASS / ☐ FAIL

---

## Step 5: Segmentation

Divide the seed ROI (isocortex/graft) into spatial segments for detailed tractography:

- **CTRL**: Isocortex → 5 anterior-posterior segments (no D-V split)
- **AP/APTR**: Graft → 9 segments (3 A-P × 3 D-V: dorsal/medial/ventral)

All segments are automatically resampled to FOD space.

In [ ]:
# ============================================================
# CTRL: Isocortex segmentation (5 A-P segments)
# ============================================================
if GROUP == "CTRL":
    print("Segmenting isocortex into 5 A-P segments...\n")
    result = run(
        f"cd {REPO_DIR} && micromamba run -n mbrain python functions/segment_isocortex_slicewise.py "
        f"--subject {SUBJECT_ID}",
        timeout=300
    )
    
    seg_dir = DWI / "isocortex_segments_v2"
    fod_dir = seg_dir / "fod_space"
    positions = ["anterior", "anterior_central", "central", "posterior_central", "posterior"]
    
    print("\n--- Isocortex segments ---")
    for pos in positions:
        for hemi in ["left", "right"]:
            check_file(seg_dir / f"isocortex_{hemi}_{pos}.nii.gz", f"{pos} {hemi}")

# ============================================================
# AP/APTR: Graft segmentation (9 segments = 3 A-P × 3 D-V)
# ============================================================
elif GROUP in ["AP", "APTR"]:
    print(f"Segmenting graft into {SEG_MODE} segments...\n")
    result = run(
        f"cd {REPO_DIR} && micromamba run -n mbrain python functions/segment_graft_slicewise.py "
        f"--subject {SUBJECT_ID} --session {SESSION_ID} --segments {SEG_MODE}",
        timeout=300
    )
    
    seg_t2w = DWI / "graft_segments_t2w"
    seg_fod = DWI / "graft_segments_fod"
    
    if SEG_MODE == 9:
        ap_labels = ["anterior", "central", "posterior"]
        dv_labels = ["dorsal", "medial", "ventral"]
    else:
        ap_labels = ["anterior", "central", "posterior"]
        dv_labels = ["dorsal", "ventral"]
    
    print(f"\n--- Graft segments (seg{SEG_MODE}) ---")
    for ap in ap_labels:
        for dv in dv_labels:
            for hemi in ["left", "right"]:
                check_file(seg_t2w / f"seg{SEG_MODE}_graft_{hemi}_{ap}_{dv}.nii.gz",
                          f"seg{SEG_MODE} {hemi} {ap}_{dv}")

### 🔍 QC Checkpoint 6: Segmentation

**Verify:** Segments should tile the seed ROI without gaps or overlaps.

In [ ]:
# Check for overlaps using the validation utility
result = run(
    f"cd {REPO_DIR} && micromamba run -n mbrain python utils/check_roi_overlap.py "
    f"--subject {SUBJECT_ID} --session {SESSION_ID} --group {GROUP}",
    timeout=120
)
print("\n👆 Verify: total unique voxels == sum of individual segments (no overlap).")

In [ ]:
# Visual QC: overlay all segments on T2w
if GROUP == "CTRL":
    seg_dir = DWI / "isocortex_segments_v2"
    cmd = f"mrview {DWI}/t2w_highres_flip.mif"
    for pos in ["anterior", "central", "posterior"]:
        cmd += f" -overlay.load {seg_dir}/isocortex_left_{pos}.nii.gz -overlay.opacity 0.4"
else:
    seg_dir = DWI / "graft_segments_t2w"
    cmd = f"mrview {DWI}/t2w_highres_flip.mif"
    for seg in ["anterior_dorsal", "central_dorsal", "posterior_dorsal"]:
        cmd += f" -overlay.load {seg_dir}/seg{SEG_MODE}_graft_left_{seg}.nii.gz -overlay.opacity 0.4"

run(f"{cmd} &", check=False)
print("\n👆 Segments should tile the seed ROI from anterior to posterior.")

**QC 6 Screenshot:**

_Paste screenshot here_

**QC 6 Result:** ☐ PASS / ☐ FAIL

---

## Step 6: Whole-Brain Tractography (1M streamlines)

Generate a dense whole-brain tractogram seeded from the brain mask.
This is the basis for passthrough extraction and SIFT2 filtering.

**⚠️ This step takes 30-90 minutes! Run and move on.**

In [ ]:
# Generate 1M whole-brain streamlines
WB_DIR = DWI / "wholebrain"
os.makedirs(WB_DIR, exist_ok=True)

WB_TCK = WB_DIR / "wholebrain_1000k.tck"
FOD = DWI / "wm_fod_norm.mif"
MASK = DWI / "mask_anat.mif"
# Fallback to mask.mif if mask_anat.mif doesn't exist
if not MASK.exists():
    MASK = DWI / "mask.mif"

n_streamlines = cfg['wholebrain_tractography']['num_streamlines']

if WB_TCK.exists():
    print(f"Whole-brain tractogram already exists: {WB_TCK}")
    run(f"tckinfo {WB_TCK} | head -10")
else:
    cmd = (f"tckgen {FOD} {WB_TCK} "
           f"-algorithm {tck['algorithm']} "
           f"-select {n_streamlines} "
           f"-step {tck['step_size']} "
           f"-angle {tck['angle']} "
           f"-cutoff {tck['cutoff']} "
           f"-minlength {tck['min_length']} "
           f"-maxlength {tck['max_length']} "
           f"-seed_image {MASK} "
           f"-mask {MASK} "
           f"-force")
    print(f"Starting tckgen with {n_streamlines} streamlines...")
    print(f"This will take 30-90 minutes.\n")
    print(f"Command:\n{cmd}\n")
    
    # Run in subprocess (long-running)
    result = run(cmd, timeout=7200)  # 2 hour timeout
    check_file(WB_TCK, "Whole-brain tractogram")

### 🔍 QC Checkpoint 7: Whole-Brain Tractogram

**Verify:** Tractogram should fill the brain volume with reasonable fiber directions.

In [ ]:
# Quick QC: tract info + visualization
run(f"tckinfo {WB_TCK}")
print("\nOpening mrview...")
run(f"mrview {DWI}/t2w_highres_flip.mif -tractography.load {WB_TCK} &", check=False)
print("\n👆 Fibers should fill the brain. Check for obvious artifacts.")

**QC 7 Screenshot:**

_Paste screenshot here_

**QC 7 Result:** ☐ PASS / ☐ FAIL

---

## Step 7: SIFT2 Weights for Whole-Brain Tractogram

Compute SIFT2 cross-section multipliers so each streamline gets a weight
proportional to how well the tractogram matches the underlying FOD.

In [ ]:
# SIFT2: compute per-streamline weights
SIFT2_WEIGHTS = WB_DIR / "wholebrain_1000k_sift2_weights.txt"
SIFT2_MU      = WB_DIR / "wholebrain_1000k_sift2_mu.txt"

if SIFT2_WEIGHTS.exists():
    print(f"SIFT2 weights already exist: {SIFT2_WEIGHTS}")
    run(f"wc -l {SIFT2_WEIGHTS}")
else:
    FOD = DWI / "wm_fod_norm.mif"
    cmd = (f"tcksift2 {WB_TCK} {FOD} {SIFT2_WEIGHTS} "
           f"-out_mu {SIFT2_MU} -force")
    print("Running SIFT2 (may take 5-15 minutes)...\n")
    result = run(cmd, timeout=1800)
    check_file(SIFT2_WEIGHTS, "SIFT2 weights")
    run(f"wc -l {SIFT2_WEIGHTS}")

---
## Step 8: Passthrough Tract Extraction

Extract streamlines that pass through specific ROI pairs from the whole-brain tractogram.
Uses `tckedit` with `-include` flags.

**Targets:** thalamus, striatum, cerebellum (with thalamus waypoint), olfactory cortex

For AP/APTR: also extracts using **shrunk** (eroded by 1 voxel) graft segments.

In [ ]:
# Extract passthrough tracts — LEFT hemisphere
print("Extracting passthrough tracts (LEFT hemisphere)...\n")
result = run(
    f"cd {REPO_DIR} && micromamba run -n mbrain python functions/extract_passthrough_tracts.py "
    f"--subject {SUBJECT_ID} --hemi left",
    timeout=1800
)

# Check output
pt_dir = WB_DIR / "passthrough_tracts"
run(f"ls -la {pt_dir}/ | head -20", check=False)

In [ ]:
# Extract passthrough tracts — RIGHT hemisphere
print("Extracting passthrough tracts (RIGHT hemisphere)...\n")
result = run(
    f"cd {REPO_DIR} && micromamba run -n mbrain python functions/extract_passthrough_tracts.py "
    f"--subject {SUBJECT_ID} --hemi right",
    timeout=1800
)

run(f"ls -la {pt_dir}/ | head -20", check=False)

In [ ]:
# ============================================================
# AP/APTR ONLY: Also extract with shrunk ROIs
# ============================================================
if GROUP in ["AP", "APTR"]:
    # First create shrunk segments if they don't exist
    print("Creating shrunk (eroded) graft segments...\n")
    run(
        f"cd {REPO_DIR} && micromamba run -n mbrain python functions/extract_passthrough_tracts.py "
        f"--subject {SUBJECT_ID} --create-shrink-segments",
        timeout=300
    )
    
    # Extract with shrunk ROIs — both hemispheres
    for hemi in ["left", "right"]:
        print(f"\nExtracting shrunk passthrough tracts ({hemi.upper()})...")
        run(
            f"cd {REPO_DIR} && micromamba run -n mbrain python functions/extract_passthrough_tracts.py "
            f"--subject {SUBJECT_ID} --hemi {hemi} --shrink",
            timeout=1800
        )
    
    shrink_dir = WB_DIR / "passthrough_tracts_shrink"
    run(f"ls -la {shrink_dir}/ | head -20", check=False)
else:
    print("Skipping shrunk extraction (CTRL subjects don't use shrunk ROIs)")

### 🔍 QC Checkpoint 8: Passthrough Tracts

**Verify:** Check a few tracts to make sure they connect the right regions.

In [ ]:
# List all extracted tracts with streamline counts
pt_dir = WB_DIR / "passthrough_tracts"
if pt_dir.exists():
    import glob
    tracts = sorted(glob.glob(str(pt_dir / "*.tck")))
    print(f"Found {len(tracts)} passthrough tract files:\n")
    for t in tracts:
        r = subprocess.run(f"tckinfo {t} -count", shell=True, capture_output=True, text=True)
        count = r.stdout.strip().split()[-1] if r.returncode == 0 else "?"
        print(f"  {Path(t).name}: {count} streamlines")
else:
    print("No passthrough tracts found yet.")

In [ ]:
# Visual QC: view a sample tract (thalamus left)
if GROUP == "CTRL":
    sample_tract = pt_dir / "isocortex_left_anterior_to_thalamus_left.tck"
    roi_overlay = BASE / "atlas" / "rois" / "fod_space" / "roi_thalamus_left.nii.gz"
else:
    # For AP/APTR use shrunk if available
    shrink_dir = WB_DIR / "passthrough_tracts_shrink"
    sample_tract = list(sorted(pt_dir.glob("*thalamus_left*.tck")))
    sample_tract = sample_tract[0] if sample_tract else None
    roi_overlay = BASE / "manual_roi" / "fod_space" / "roi_Thalamus_left_flip_fod.mif"

if sample_tract and Path(sample_tract).exists():
    cmd = (f"mrview {DWI}/t2w_highres_flip.mif "
           f"-overlay.load {roi_overlay} -overlay.opacity 0.4 "
           f"-tractography.load {sample_tract} &")
    run(cmd, check=False)
    print(f"\n👆 Showing: {Path(sample_tract).name}")
    print("   Fibers should connect seed region to thalamus.")
else:
    print("No sample tract found for visualization.")

**QC 8 Screenshot:**

_Paste screenshot here_

**QC 8 Result:** ☐ PASS / ☐ FAIL

---

## Step 9: MATLAB AFQ Cleaning

Remove outlier streamlines using MATLAB AFQ_removeFiberOutliers.
This uses statistical criteria (distance and length) to remove spurious fibers.

**Parameters** (from config1.yaml):
- `max_dist`: 3 (Gaussian distance threshold)
- `max_len`: 3 (length deviation threshold)
- `num_nodes`: 100 (resampling nodes)
- `min_streamlines`: 20 (minimum to attempt cleaning)

In [ ]:
# Print cleaning parameters
print("MATLAB AFQ Cleaning Parameters:")
for k, v in cfg['matlab_cleaning'].items():
    print(f"  {k}: {v}")

In [ ]:
# Clean passthrough tracts
print("Cleaning passthrough tracts with MATLAB AFQ...\n")
print("(This runs MATLAB in batch mode — may take 5-20 minutes)\n")

result = run(
    f"cd {REPO_DIR} && micromamba run -n mbrain python functions/clean_tracts.py "
    f"--subject {SUBJECT_ID} --passthrough",
    timeout=3600
)

# Count cleaned files
import glob
n_cleaned = len(glob.glob(str(pt_dir / "*_cleaned.tck")))
print(f"\nCleaned files created: {n_cleaned}")

In [ ]:
# ============================================================
# AP/APTR ONLY: Also clean shrunk tracts
# ============================================================
if GROUP in ["AP", "APTR"]:
    print("Cleaning shrunk passthrough tracts...\n")
    result = run(
        f"cd {REPO_DIR} && micromamba run -n mbrain python functions/clean_tracts.py "
        f"--subject {SUBJECT_ID} --passthrough --shrink",
        timeout=3600
    )
    shrink_dir = WB_DIR / "passthrough_tracts_shrink"
    n_cleaned_shrink = len(glob.glob(str(shrink_dir / "*_cleaned.tck")))
    print(f"\nCleaned shrunk files: {n_cleaned_shrink}")
else:
    print("Skipping shrunk cleaning (CTRL)")

### 🔍 QC Checkpoint 9: Cleaned vs Raw Tracts

**Verify:** Cleaned tracts should retain the main bundle but remove outliers.

In [ ]:
# Compare raw vs cleaned for one tract
import glob

# Find a cleaned tract to compare
cleaned_files = sorted(glob.glob(str(pt_dir / "*_cleaned.tck")))
if cleaned_files:
    cleaned = Path(cleaned_files[0])
    raw = cleaned.parent / cleaned.name.replace("_cleaned.tck", ".tck")
    
    print(f"Comparing: {raw.name}")
    run(f"tckinfo {raw} -count")
    print(f"Cleaned: {cleaned.name}")
    run(f"tckinfo {cleaned} -count")
    
    # Open both side by side
    run(f"mrview {DWI}/t2w_highres_flip.mif "
        f"-tractography.load {cleaned} &", check=False)
    print("\n👆 Cleaned tract should be a tight, clean bundle.")
else:
    print("No cleaned tracts found yet.")

**QC 9 Screenshot:**

_Paste screenshot here_

**QC 9 Result:** ☐ PASS / ☐ FAIL

---

## Step 10: Generate Metrics Report

Compute tract metrics and generate CSV + HTML report:
- Streamline count (from cleaned tracts)
- SIFT2 connectivity (from raw tracts matched to whole-brain weights)
- Mean AFD (Apparent Fiber Density) along tracts

**Note:** AP/APTR automatically uses shrunk tracts for the report.

In [ ]:
# Generate metrics report
import datetime
version = f"v{datetime.datetime.now().strftime('%Y%m%d')}_{SUBJECT_ID}"

print(f"Generating metrics report (version: {version})...\n")
result = run(
    f"cd {REPO_DIR} && micromamba run -n mbrain python functions/generate_tract_metrics_report.py "
    f"--cleaned --subject {SUBJECT_ID} --version {version}",
    timeout=600
)

# Check outputs
reports_dir = REPO_DIR / "reports"
check_file(reports_dir / f"tract_metrics_{version}.csv", "CSV report")
check_file(reports_dir / f"tract_metrics_report_{version}.html", "HTML report")

In [ ]:
# Preview the CSV report
import pandas as pd

csv_path = REPO_DIR / "reports" / f"tract_metrics_{version}.csv"
if csv_path.exists():
    df = pd.read_csv(csv_path)
    print(f"Report: {len(df)} rows\n")
    print("Columns:", list(df.columns))
    print(f"\nSummary for {SUBJECT_ID}:")
    display(df[df['subject'] == SUBJECT_ID].to_string(index=False))
else:
    print("Report CSV not found.")

---
## Step 11: Final Visualization & Summary

View the complete results for this subject.

In [ ]:
# Summary table: all tracts for this subject
print(f"\n{'='*80}")
print(f" PIPELINE COMPLETE: {SUBJECT_ID} ({GROUP})")
print(f"{'='*80}\n")

# Count outputs
import glob

pt_dir = WB_DIR / "passthrough_tracts"
raw_tracts = glob.glob(str(pt_dir / "*.tck")) if pt_dir.exists() else []
raw_tracts = [t for t in raw_tracts if "_cleaned" not in t]
cleaned_tracts = glob.glob(str(pt_dir / "*_cleaned.tck")) if pt_dir.exists() else []

print(f"Whole-brain tractogram:  {'✅' if WB_TCK.exists() else '❌'}")
print(f"SIFT2 weights:          {'✅' if SIFT2_WEIGHTS.exists() else '❌'}")
print(f"Raw passthrough tracts:  {len(raw_tracts)}")
print(f"Cleaned tracts:          {len(cleaned_tracts)}")

if GROUP in ["AP", "APTR"]:
    shrink_dir = WB_DIR / "passthrough_tracts_shrink"
    shrink_raw = glob.glob(str(shrink_dir / "*.tck")) if shrink_dir.exists() else []
    shrink_raw = [t for t in shrink_raw if "_cleaned" not in t]
    shrink_cleaned = glob.glob(str(shrink_dir / "*_cleaned.tck")) if shrink_dir.exists() else []
    print(f"Shrunk raw tracts:       {len(shrink_raw)}")
    print(f"Shrunk cleaned tracts:   {len(shrink_cleaned)}")

print(f"\nReport: reports/tract_metrics_{version}.csv")
print(f"HTML:   reports/tract_metrics_report_{version}.html")
print(f"\n{'='*80}")

In [ ]:
# Final comprehensive visualization
# Open mrview with T2w + best tracts

if GROUP == "CTRL":
    seg_roi = BASE / "atlas" / "rois" / "fod_space" / "roi_isocortex_left.nii.gz"
    target_roi = BASE / "atlas" / "rois" / "fod_space" / "roi_thalamus_left.nii.gz"
else:
    seg_roi = BASE / "manual_roi" / "fod_space" / "roi_Graft_left_flip_fod.mif"
    target_roi = BASE / "manual_roi" / "fod_space" / "roi_Thalamus_left_flip_fod.mif"

# Find all cleaned tracts
cleaned = sorted(glob.glob(str(pt_dir / "*left*_cleaned.tck")))

cmd = (f"mrview {DWI}/t2w_highres_flip.mif "
       f"-overlay.load {seg_roi} -overlay.opacity 0.3 "
       f"-overlay.load {target_roi} -overlay.opacity 0.3")

for t in cleaned[:6]:  # Load up to 6 tracts
    cmd += f" -tractography.load {t}"

run(f"{cmd} &", check=False)
print("\n👆 Final visualization with all left-hemisphere cleaned tracts.")

**Final Screenshot:**

_Paste screenshot here_

---

## QC Summary

| Step | Check | Result |
|------|-------|--------|
| 1 | T2w orientation (dorsal up, anterior right) | ☐ PASS / ☐ FAIL |
| 2 | DWI-T2w registration (b0 aligned with T2w) | ☐ PASS / ☐ FAIL |
| 3 | ROIs on correct anatomy (L on L, R on R) | ☐ PASS / ☐ FAIL |
| 4 | FOD quality (directional glyphs in WM) | ☐ PASS / ☐ FAIL |
| 5 | ROIs in FOD space (aligned on FOD grid) | ☐ PASS / ☐ FAIL |
| 6 | Segmentation (no overlaps, tiles seed ROI) | ☐ PASS / ☐ FAIL |
| 7 | Whole-brain tractogram (fills brain volume) | ☐ PASS / ☐ FAIL |
| 8 | Passthrough tracts (connect correct regions) | ☐ PASS / ☐ FAIL |
| 9 | Cleaned tracts (tight bundles, outliers removed) | ☐ PASS / ☐ FAIL |

**Processed by:** _Your name_  
**Date:** _Date_  
**Notes:** _Any issues or observations_